# Histograms – Exercises
_Mikołaj Leszczuk_

![CC-BY](https://i.creativecommons.org/l/by/4.0/88x31.png)


## Goal
Short, practical exercises to consolidate histogram skills. Focus on: CDF & percentiles, Contrast Limited Adaptive Histogram Equalization (CLAHE, local), color-space histograms (Hue), and histogram **distance** metrics.

In [ ]:
# Setup
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

plt.rcParams['figure.figsize'] = (6,4)

## Exercise 1 - CDF & Percentile Stretch
**Idea:** Use the **cumulative histogram (CDF)** to compute low/high **percentiles** (e.g., 2% and 98%) and perform **contrast stretching** by clipping to these bounds and linearly rescaling.

**Steps:**

1. Load a grayscale image.

In [ ]:
# Your code here
img_path = 'Unequalized_Hawkes_Bay_NZ.jpg'
img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)

In [ ]:
plt.imshow(img, cmap='gray')

2. Compute its histogram and CDF.

In [ ]:
# histogram & cdf
hist = cv2.calcHist([img],[0],None,[256],[0,256]).ravel()
cdf = np.cumsum(hist)
cdf_norm = cdf / cdf[-1]

3. Find intensity values at **p2** and **p98** percentiles.

In [ ]:
# percentiles
p2 = np.searchsorted(cdf_norm, 0.02)
p98 = np.searchsorted(cdf_norm, 0.98)
print(p2, p98)

4. Clip and rescale the image from `[p2, p98]` to `[0, 255]`.

In [ ]:
# clip & stretch
img_clipped = np.clip(img, p2, p98)
stretched = ((img_clipped - p2) / max(1, (p98 - p2)) * 255).astype(np.uint8)

5. Plot: original vs stretched image and their histograms (with CDF overlay).

In [ ]:
fig, axes = plt.subplots(1,2)
axes[0].imshow(img, cmap='gray'); axes[0].set_title('Original'); axes[0].axis('off')
axes[1].imshow(stretched, cmap='gray'); axes[1].set_title('Percentile-stretched'); axes[1].axis('off')
plt.show()

In [ ]:
fig, axes = plt.subplots(1,2)
axes[0].plot(hist); ax2=axes[0].twinx(); ax2.plot(cdf_norm, linestyle='--'); axes[0].set_title('Original: hist & CDF')
hist2 = cv2.calcHist([stretched],[0],None,[256],[0,256]).ravel()
axes[1].plot(hist2); ax3=axes[1].twinx(); ax3.plot(np.cumsum(hist2)/hist2.sum(), linestyle='--'); axes[1].set_title('Stretched: hist & CDF')
plt.show()

**Questions:** What happens to very dark/bright tails? Why do percentiles often work better than naive min-max?

## Exercise 2 - Local Histogram Enhancement (CLAHE)
**Idea:** Apply **Contrast Limited Adaptive Histogram Equalization (CLAHE)** on the **luminance** channel to enhance local contrast without blowing out global tones.

**Steps:**

1. Load a **color** image.

In [ ]:
img_path = 'SunLou2.jpg'
bgr = cv2.imread(img_path)

In [ ]:
plt.imshow(cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB))

2. Convert to **LAB** (or YCrCb), take L (luma) channel.

In [ ]:
lab = cv2.cvtColor(bgr, cv2.COLOR_BGR2LAB)
L, A, B = cv2.split(lab)

3. Apply `cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))`.

In [ ]:
clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
L_eq = clahe.apply(L)

4. Merge back and compare with original.

In [ ]:
lab_eq = cv2.merge([L_eq, A, B])
bgr_eq = cv2.cvtColor(lab_eq, cv2.COLOR_LAB2BGR)

fig, axes = plt.subplots(1,2)
axes[0].imshow(cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)); axes[0].set_title('Original'); axes[0].axis('off')
axes[1].imshow(cv2.cvtColor(bgr_eq, cv2.COLOR_BGR2RGB)); axes[1].set_title('CLAHE (L channel)'); axes[1].axis('off')
plt.show()

5. Compare histograms of the L channel (before vs after).

In [ ]:
hist_L = cv2.calcHist([L],[0],None,[256],[0,256]).ravel()
hist_Le = cv2.calcHist([L_eq],[0],None,[256],[0,256]).ravel()
plt.plot(hist_L, label='L original'); plt.plot(hist_Le, label='L CLAHE'); plt.legend(); plt.title('L channel histograms'); plt.show()

## Exercise 3 - Color Dominance via **Hue Histogram**
**Idea:** Use **Hue** histograms to detect dominant colors robust to illumination changes.

**Steps:**

1. Convert image to **HSV**.

In [ ]:
img_path = 'SunLou2.jpg'
bgr = cv2.imread(img_path)

# Convert to HSV and compute Hue histogram
hsv = cv2.cvtColor(bgr, cv2.COLOR_BGR2HSV)

In [ ]:
# Show original image
plt.subplot(1, 2, 1)
plt.imshow(cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB))
plt.title('Original image')
plt.axis('off')

2. Compute histogram for **H channel** only (bins=180).

In [ ]:
H = hsv[:, :, 0]
histH = cv2.calcHist([H], [0], None, [180], [0, 180]).ravel()

3. Normalize histogram and identify top-3 dominant hue bins.

In [ ]:
histH = histH / (histH.sum() + 1e-8)
top3 = histH.argsort()[-3:][::-1]
print('Top-3 hue bins:', top3)

4. How shadows/brightness affect Hue vs grayscale histograms?

In [ ]:
# Visualization
plt.figure(figsize=(10, 4))

In [ ]:
# 1. Plot Hue histogram with top-3 bins marked
plt.subplot(1, 2, 2)
plt.plot(histH, color='gray')
for h in top3:
    plt.axvline(int(h), color='red', linestyle='--')
plt.title('Hue histogram (normalized)')
plt.xlabel('Hue bin (0–179)')
plt.ylabel('Frequency')

plt.tight_layout()
plt.show()

In [ ]:
# 2. Display color swatches for top-3 Hues
swatch = np.zeros((50, 60 * 3, 3), dtype=np.uint8)
for i, h in enumerate(top3):
    hsv_color = np.uint8([[[int(h), 255, 255]]])  # max saturation and brightness
    bgr_color = cv2.cvtColor(hsv_color, cv2.COLOR_HSV2BGR)[0, 0]
    swatch[:, i * 60:(i + 1) * 60, :] = bgr_color

plt.imshow(cv2.cvtColor(swatch, cv2.COLOR_BGR2RGB))
plt.title(f'Dominant colors (Hue bins): {list(map(int, top3))}')
plt.axis('off')
plt.show()

## Exercise 4 - Histogram **Distance**
**Idea:** Compare two images by histogram **distance**. Use **Bhattacharyya** or **Chi-square** to quantify similarity.

**Steps:**

1. Load two grayscale images with similar size/content.

In [ ]:
# --- Load images ---
p1 = 'Grayscale_Image.tif.jpg'
p2 = 'Output_image.tif.jpg'
im1 = cv2.imread(p1, cv2.IMREAD_GRAYSCALE)
im2 = cv2.imread(p2, cv2.IMREAD_GRAYSCALE)

In [ ]:
# --- Visualization ---
plt.figure(figsize=(10, 5))

In [ ]:
# Show both images side by side
plt.subplot(1, 2, 1)
plt.imshow(im1, cmap='gray')
plt.title('Image 1')
plt.axis('off')

plt.subplot(1, 2, 2)
plt.imshow(im2, cmap='gray')
plt.title('Image 2')
plt.axis('off')

plt.suptitle('Input Images')
plt.tight_layout()
plt.show()

2. Compute normalized histograms (256 bins).

In [ ]:
# --- Compute normalized histograms ---
h1 = cv2.calcHist([im1], [0], None, [256], [0, 256]).astype(np.float32)
h2 = cv2.calcHist([im2], [0], None, [256], [0, 256]).astype(np.float32)
h1 /= (h1.sum() + 1e-8)
h2 /= (h2.sum() + 1e-8)

3. Compute distances: Bhattacharyya (`cv2.HISTCMP_BHATTACHARYYA`) and Chi-square (`cv2.HISTCMP_CHISQR`).

In [ ]:
# --- Compare histograms ---
bh = cv2.compareHist(h1, h2, cv2.HISTCMP_BHATTACHARYYA)
chi = cv2.compareHist(h1, h2, cv2.HISTCMP_CHISQR)

print(f'Bhattacharyya distance: {bh:.6f}')
print(f'Chi-square distance: {chi:.6f}')
print("(For two identical images, both values should be ≈ 0)")

4. Interpret: lower Bhattacharyya means more similar distributions.

In [ ]:
# Show histograms
plt.figure(figsize=(6, 4))
plt.plot(h1, label='Image 1', color='blue')
plt.plot(h2, label='Image 2', color='orange', linestyle='--')
plt.title('Normalized Histograms')
plt.xlabel('Gray level')
plt.ylabel('Frequency')
plt.legend()
plt.show()